# 🧠 Variational Quantum Classifier (VQC) Deep Dive

This notebook explores the design, expressibility, and training of Variational Quantum Circuits (VQC)
for binary and multiclass classification tasks using **QuantumShield-FL**.

## Contents
1. [Quantum Encodings](#encodings)
2. [Hardware-Efficient & Strongly Entangling Ansätze](#ansatz)
3. [Exact Gradients via the Parameter-Shift Rule](#paramshift)
4. [Training VQC on Non-Linear Benchmark Datasets](#training)
5. [Quantum Decision Boundaries Visualization](#boundaries)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.preprocessing import StandardScaler

from src.quantum.qml import VQC, QuantumDataEncoder

# Plot style
plt.rcParams['figure.facecolor'] = '#0f172a'
plt.rcParams['axes.facecolor'] = '#1e293b'
plt.rcParams['text.color'] = '#f8fafc'
plt.rcParams['axes.labelcolor'] = '#cbd5e1'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['grid.color'] = '#334155'

print('⚛️ Quantum ML Module ready!')

## 1. Non-linear Data Generation (Moons Dataset)

Classical linear models fail on interlocking half-moons. We evaluate whether a 2-qubit VQC
with strongly entangling layers can map the data into a higher-dimensional Hilbert space
and classify it with high fidelity.

In [ ]:
X, y = make_moons(n_samples=120, noise=0.15, random_state=42)
X = StandardScaler().fit_transform(X) * (np.pi / 2)

plt.figure(figsize=(7, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='#38bdf8', label='Class 0', edgecolors='none', s=50)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='#f43f5e', label='Class 1', edgecolors='none', s=50)
plt.title('Non-Linear Two Moons Dataset', fontsize=12, fontweight='bold')
plt.xlabel('Feature 0 (scaled to radians)')
plt.ylabel('Feature 1 (scaled to radians)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Train Variational Quantum Classifier (VQC)

We instantiate a VQC with `n_qubits=2`, `n_layers=3`, and `ansatz='strongly_entangling'`.

In [ ]:
vqc = VQC(n_qubits=2, n_layers=3, ansatz='strongly_entangling', learning_rate=0.08)
print(f'Trainable parameters count: {vqc.params.size}')

print('🚀 Training VQC with Adam and Parameter-Shift Rule...')
history = vqc.fit(X, y, n_epochs=25, verbose=True)

final_acc = vqc.score(X, y)
print(f'\n🎯 Training Accuracy: {final_acc*100:.2f}%')

## 3. Quantum Decision Boundary Visualization

In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))

grid_points = np.c_[xx.ravel(), yy.ravel()]
Z = vqc.predict_proba(grid_points)[:, 1].reshape(xx.shape)

plt.figure(figsize=(8, 6))
contour = plt.contourf(xx, yy, Z, levels=20, cmap='coolwarm', alpha=0.7)
plt.colorbar(contour, label='P(Class = 1)')
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='#0284c7', label='Class 0', edgecolors='white', s=50)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='#e11d48', label='Class 1', edgecolors='white', s=50)
plt.title('Quantum Decision Boundary (VQC)', fontsize=13, fontweight='bold')
plt.xlabel('Feature 0')
plt.ylabel('Feature 1')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()
print('✅ Quantum decision surface plotted!')